# 1D FWI implementation via ODIL

In this notebook, I will extend what I have learnt of ODIL from reproducing Figure 1 in the [`/reproduction`](./../reproduction/) notebooks. Our task will be to jointly optimise for an amplitude field and a wavespeed field using the ODIL framework. This will involve introducing a source term, the concept of 'receievers' (discrete observation points), and a data misfit.

To start, we can update our `Wavefield` class to include a wavespeed model in its data vector. It is important we keep the wavespeed and ampltiude data together. If we do not perform a joint optimisation and instead optimise the two field separately, the wave equation solve will be chasing an expired wavespeed model. 

In [4]:
from dataclasses import dataclass, field
from typing import Tuple
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
@dataclass
class Wavefield:
    # spatial data
    x_0: float = -1.0
    x_I: float = 1.0
    I: int = 25
    L: float = field(init=False)
    dx: float = field(init=False)

    # temporal data
    t_0: float = 0.0
    t_N: float = 1.0
    N: int = 25
    T: float = field(init=False)
    dt: float = field(init=False)

    # actual data
    init_amplitude: np.ndarray | None = None  # data to initialise with
    init_wavespeed: np.ndarray | None = None
    wavespeed_idx: float = N * I # where wavespeed starts in internal data

    _data: np.ndarray = field(init=False)  # internal data
    _shape: Tuple[int, int] = field(init=False)  # length of domain object

    def __post_init__(self):
        # compute lengths and spacings
        self.L = self.x_I - self.x_0
        self.dx = self.L / (self.I - 1)

        self.T = self.t_N - self.t_0
        self.dt = self.T / (self.N - 1)

        self._shape = (self.N, self.I)  # t, x
        if self.init_amplitude is None:
            amplitude = np.zeros(shape=(self._shape), dtype=float).flatten()
        if self.init_wavespeed is None:
            wavespeed = np.zeros(shape=(self.I))  # not time varying 
        else:
            amplitude = np.asarray(self.amplitude, dtype=float).flatten()
            wavespeed = np.asarray(self.wavespeed, dtype=float).flatten()

        # concatenate the two
        # note: wavespeed starts at NxI
        self._data = np.concat((amplitude, wavespeed))

    # convention: space index i, time index n
    def __getitem__(self, key):
        i, n = key
        return self._data[i + self.I * n]

    def __setitem__(self, key, value):
        i, n = key
        self._data[i + self.I * n] = value

    def __sub__(self, other):
        return self._data - other._data

    def __pow__(self, other):
        return self._data**other

    @property
    def data(self):
        return self._data
    
    @property
    def amplitude(self):
        return self._data[:self.wavespeed_idx]
    
    @property
    def wavespeed(self):
        return self._data[self.wavespeed_idx:]

    @property
    def shape(self):
        return self._shape

    def show(self, title: str = "Exact solution") -> None:
        fig, ax = plt.subplots(figsize=(8, 5))
        umax = np.abs(self._data).max()
        im = ax.imshow(
            self.amplitude.reshape(self.N, self.I),
            extent=(self.x_0, self.x_I, self.t_N, self.t_0),  # [xmin, xmax, tmax, tmin]
            cmap="RdBu_r",
            vmin=-umax,
            vmax=umax,
            aspect=4,
        )

        plt.colorbar(im, ax=ax, label="u(x, t)")
        ax.set_xlabel("x")
        ax.set_ylabel("t")
        ax.set_title(title)
        plt.tight_layout()
        plt.show()